In [ ]:
!pip install -q transformers==4.51.1 kagglehub

In [ ]:
import kagglehub

DATA_DIR = kagglehub.dataset_download("sattamjaltwaim/ioai-cuties")
print(f"Dataset path: {DATA_DIR}")

In [ ]:
import torch
from torch.nn import functional as F
from transformers import AutoProcessor, CLIPModel
from PIL import Image
import numpy as np
import os
import base64
import io
import pandas as pd
from tqdm import tqdm

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
processor = AutoProcessor.from_pretrained('openai/clip-vit-base-patch16')
model = CLIPModel.from_pretrained('openai/clip-vit-base-patch16').to(device).eval()
print(f"Device: {device}")

In [ ]:
# Paths to validation images, masks, and test images
VAL_IMGS = f'{DATA_DIR}/val_imgs'
VAL_MASKS = f'{DATA_DIR}/val_masks'
TEST_IMGS = f'{DATA_DIR}/test_imgs'

# Load the breed names that CLIP will match against
class_names = open(f'{DATA_DIR}/breeds.txt').read().strip().split('\n')
print(f"Breeds: {len(class_names)}")
print(f"Val images: {len(os.listdir(VAL_IMGS))}")
print(f"Test images: {len(os.listdir(TEST_IMGS))}")

In [ ]:
# Encode every breed name into CLIP text space
tokens = processor(text=class_names, images=None, return_tensors='pt', padding=True).to(device)
with torch.no_grad():
    text_embs = model.get_text_features(**tokens)
    text_embs = F.normalize(text_embs, dim=-1)
print(f"Text embeddings: {text_embs.shape}")

In [ ]:
# CLIP ViT-B/16 splits each image into a 14x14 grid of patches.
# Each patch becomes a token in the vision transformer.
# We project every patch token into CLIP space, compute cosine
# similarity with the detected breed, and threshold at the mean.

NUM_PATCHES = 14  # 224 / 16


def segment_image(img):
    img_h, img_w = np.array(img).shape[:2]
    pixel_values = processor(images=img, return_tensors='pt')['pixel_values'].to(device)

    with torch.no_grad():
        vision_out = model.vision_model(pixel_values=pixel_values)
        hidden = vision_out.last_hidden_state  # (1, 197, 768) -- 1 CLS + 196 patches

        # Project all tokens to shared CLIP space
        hidden = model.vision_model.post_layernorm(hidden)
        projected = model.visual_projection(hidden)  # (1, 197, 512)
        projected = F.normalize(projected, dim=-1)

        # Use the CLS token to figure out which breed this is
        cls_token = projected[:, 0, :]  # (1, 512)
        breed_idx = (cls_token @ text_embs.T).argmax(dim=1).item()
        breed_emb = text_embs[breed_idx]

        # Cosine similarity between each patch and the breed embedding
        patches = projected[:, 1:, :]  # (1, 196, 512)

    sim = (patches @ breed_emb.unsqueeze(-1)).squeeze()  # (196,)
    sim_map = sim.reshape(1, 1, NUM_PATCHES, NUM_PATCHES)

    # Upscale the 14x14 map back to the original image size
    sim_map = F.interpolate(sim_map, size=(img_h, img_w), mode='bilinear', align_corners=False)
    sim_map = sim_map.squeeze().cpu().numpy()

    # Threshold at the mean -- above mean is animal, below is background
    mask = (sim_map > sim_map.mean()).astype(np.uint8) * 255
    return mask

In [ ]:
def binary_iou(mask1, mask2):
    """Intersection over union for two binary masks."""
    intersection = np.logical_and(mask1 == 1, mask2 == 1).sum()
    union = np.logical_or(mask1 == 1, mask2 == 1).sum()
    return intersection / union if union > 0 else 0.0

In [ ]:
# Check how well we do on the 20 validation images
ious = []

for name in tqdm(sorted(os.listdir(VAL_IMGS))):
    if not name.endswith('.jpg'):
        continue

    img = Image.open(os.path.join(VAL_IMGS, name))
    gt = np.array(Image.open(os.path.join(VAL_MASKS, name.replace('.jpg', '.png')))) // 255

    pred = segment_image(img) // 255
    ious.append(binary_iou(pred, gt))

print(f"Val mean IoU: {np.mean(ious):.4f}")

In [ ]:
# Generate masks for every test image
test_masks = {}

for name in tqdm(sorted(os.listdir(TEST_IMGS))):
    if not name.endswith('.jpg'):
        continue

    img = Image.open(os.path.join(TEST_IMGS, name))
    test_masks[int(name.replace('.jpg', ''))] = segment_image(img)

print(f"Generated {len(test_masks)} masks")

In [ ]:
# ============================================
# DO NOT MODIFY -- Submission Generator
# ============================================
def image_to_base64(image, fmt='PNG'):
    buf = io.BytesIO()
    image.save(buf, format=fmt)
    return base64.b64encode(buf.getvalue()).decode('utf-8')

ids, b64_masks = [], []
for img_id in sorted(test_masks.keys()):
    mask_img = Image.fromarray(test_masks[img_id].astype(np.uint8)).convert('L')
    ids.append(img_id)
    b64_masks.append(image_to_base64(mask_img))

submission = pd.DataFrame({'img_id': ids, 'mask': b64_masks})
submission.to_csv('submission.csv', index=False)
print(f"Saved submission.csv with {len(submission)} rows")